# Nutosis

It is October 2018. The squirrels in Central Park are running into a problem and we need your help.

You are intentionally given very little information thus far. You must communicate with your client (me) for additional information as necessary. But also make sure that your communications are efficient, thought out, and not redundant as your client might get frustrated and "fire" you (this only applies to getting information from your client, this does not necessary apply to asking for help with the actual project itself - you should continuously ask questions for getting help).

For this project you must:
 * go through most steps in the checklist (see below for details).
 * write responses for all items however sometimes the item will simply be "does not apply". Some of the parts are a bit more nebulous and you simply show that you have done things in general (and the order doesn't really matter).
 * keep your progress and thoughts organized in this document and use formatting as appropriate (using Markdown to add headers and sub-headers for each major part).

Notes about individual steps:
 * **Frame the problem**: Complete all parts in this notebook.
 * **Get the data**: Complete all parts in this notebook.
   * You must include at least 2 additional data sources.
   * Do not commit any data, instead the code should fetch and cache the data.
   * Must create `train_set.parquet` and `test_set.parquet` files. These files should be committed (but no other data should be committed).
 * **Explore the data**: Complete all parts in the Exploration notebook.
 * **Prepare the data**: Complete all parts in the Preparation notebook.
   * Must create `preprocessor.pkl` file that takes the train or test set and outputs the dataset ready to train on.
 * **Short-List Promising Models**: Complete all parts in the Training notebook.
   * You must check at least 6 base models and select at least 3 to continue with.
   * Anything that you learn here that changes your data preparation should be noted here but actually implemented in the Preparation notebook.
 * **Fine-Tune the System**: Complete all parts in the Training notebook.
   * You must fine-tune at least 3 models.
   * You must try tuning at least 1 ensemble model in addition to the 3.
   * Each fine-tuning must include at least 10 hyperparameters across a wide range of values. At least some must be in the data preprocessing and some must be in the learning model.
   * Must create `model.pkl` that has the final, best, model selected along with the preprocessor in it.
 * **Present Solution**: Complete all parts in the Presentation notebook.
   * Your "presentation" will be done as information written in that notebook, no slides or anything like that. It should include the best summary plots/graphics/data points.
 * **Launch**: Do not complete at all.

You must submit the train/test parquet files and a pickled preprocessor and final model along with these notebooks.

# Frame the Problem and Look at the Big Picture

1. Define the objective in business terms.
The wildlife conservation team needs a way to catch sick squirrels early even though they are short staffed. Our goal is to use volunteer sighting reports to flag squirrels that are likely sick, so staff can focus their limited time on the squirrels most worth capturing and testing instead of checking at random. Success means catching more sick squirrels early with the same number of staff hours and wasting fewer captures on healthy ones.

2. How will your solution be used?
Volunteers will keep submitting sightings through the app, which comes out to around 40 to 50 answered reports a day. Every few hours the model will score the newest batch of reports and produce a ranked list of squirrels that are likely sick, along with where and when they were seen. Staff can use that list to decide where to send people for capture and testing. Once test results come back, they get fed into the system as new samples so the model keeps improving over time.

3. What are the current solutions/workarounds (if any)?
The wildlife conservation team uses reported sightings and decide if its a credible source. If the team deems it as credible they go out, test the squirrels immune system and euthanize the squirrel at their lab if they show signs of Nutosis. 

4. How should you frame this problem (supervised/unsupervised, online/offline, …)?
This is a supervised learning problem because we have labeled outcomes from captures and testing that tell us whether a squirrel was sick. We would start with binary classification (sick or healthy) and could move to a multi class approach since Nutosis is evolving so (healthy, early stage, and advanced). Since new reports keep coming in, it fits a online batch setup where the model scores and retrains every few hours. 

5. How should performance be measured? Is the performance measure aligned with the business objective?

6. What would be the minimum performance needed to reach the business objective?

7. What are comparable problems? Can you reuse experience or tools?
We can reuse certain tools like matplotlib, pandas, numpy and scikit-learn.

8. Is human expertise available?
Since Nutosis is a new virus, the only way to diagnose a squirrel is by testing its immune system. The staff do not have the expertise to spot signs of Nutosis from a squirrel's appearance.

9. How would you solve the problem manually?
Staff would go through each volunteer report as it comes in, remove duplicate sightings of the same squirrel, and look for patterns like clusters of reports in the same area or unusual behavior. Based on that, they would pick which squirrels to capture, test their immune systems, and record whether each one was sick. Over time they would keep track of where positive cases showed up and use that to decide where to look next. 

10. List the assumptions you (or others) have made so far. Verify assumptions if possible.


# Get the Data

1. List the data you need and how much you need
- diseased.csv - all will be needed
- 2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104.csv - all will be needed 
2. Find and document where you can get that data
- diseased.csv - provided by our client
- 2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104 - Nyc open data
3. Get access authorizations
- diseased.csv - we have access from our client 
- 2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104
4. Create a workspace (with enough storage space) <br>
This notebook. 

5. Get the data
- Download and helper functions were provided for all csv files
6. Convert the data to a format you can easily manipulate (without changing the data itself)
- diseased.csv - is a csv file 
- 2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104.csv - is a csv file 
7. Ensure sensitive information is deleted or protected (e.g. anonymized)
- Data shows no sensitive information

In [4]:
%pip install kagglehub scikit-learn matplotlib numpy ipywidgets pandas pyarrow
import kagglehub
from kagglehub import KaggleDatasetAdapter
import os
import pandas as pd
import numpy as np
import shutil
from sklearn.model_selection import train_test_split


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Functions

In [5]:
URL = "jamellalvarez/project2nutosisdataset"

def list_dataset_files():
    """Prints the files in the Kaggle dataset so you know which names to use."""
    cache_dir = kagglehub.dataset_download(URL)
    files = sorted(os.listdir(cache_dir))
    for f in files:
        print(f)
    return files

def get_local_file(filename):
    """
    Checks if filename exists locally. If not, fetches it from
    the Kaggle cache and copies it to your project directory.
    """
    if not os.path.exists(filename):
        print(f"Fetching {filename} from Kaggle...")
        cache_dir = kagglehub.dataset_download(URL)
        cached_file_path = os.path.join(cache_dir, filename)
        shutil.copy(cached_file_path, filename)
    return filename

def read_squirrel_data():
    """Loads the squirrel census sightings from the local directory."""
    return pd.read_csv(get_local_file("2018_Central_Park_Squirrel_Census_-_Squirrel_Data_20261104.csv"))


def read_sickness_data():
    """Loads the Nutosis test results from the local directory."""
    return pd.read_csv(get_local_file("diseased.csv"))


# Data 

In [ ]:
squirrel_data = read_squirrel_data()
sickness_data = read_sickness_data()

squirrel_data.shape
# sickness_data.shape

# squirrel_data.head()
# sickness_data.head()


(3023, 31)

# Train and Test Split

The reason we went for this test/train split ...